# All GPU runs in one session (issues #10, #12, #13)
Order matters: the paraphrase files must exist before the zero-shot and RoBERTa `paraphrase*` settings can be scored.

1. paraphrase attack (#13)  2. zero-shot ppl / logrank / fastdetectgpt (#12)  3. RoBERTa (#10)  4. metrics  5. one zip to download.

**Settings:** GPU T4, Internet on, dataset `team20-m4-processed` attached. Use **Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed; then download the zip from the version's Output tab. No logic in this notebook: every cell calls a script in `src/`.

In [ ]:
# Use -b d-attack until PR #21 is merged; switch to -b main afterwards.
!git clone -b d-attack https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits

## 1. Paraphrase attack (#13)

In [ ]:
!python -m src.paraphrase

## 2. Make `data/processed` writable so every script can read `paraphrased.parquet`
`/kaggle/input` is read-only, so link the main dataset file and copy the new parquet next to it. Then re-check that every id in every split file (now including `paraphrase*`) exists.

In [ ]:
!SRC="$(find /kaggle/input -name m4_en.parquet | head -1)"; rm -f data/processed; mkdir -p data/processed; ln -s "$SRC" data/processed/m4_en.parquet; cp outputs/paraphrase/paraphrased.parquet data/processed/paraphrased.parquet
!ls -l data/processed/
!python -m src.data --check-splits

## 3. Zero-shot (#12): scores every setting, including `paraphrase_orig`, `paraphrase`, `paraphrase_human`

In [ ]:
!python -m src.zeroshot --method ppl

In [ ]:
!python -m src.zeroshot --method logrank

In [ ]:
!python -m src.zeroshot --method fastdetectgpt

### Checkpoint zip (download this first if the session is cut short)

In [ ]:
!zip -rq /kaggle/working/checkpoint_zeroshot_paraphrase.zip results/ppl results/logrank results/fastdetectgpt results/paraphrase_quality.json splits/paraphrase.json splits/paraphrase_human.json outputs/paraphrase/paraphrased.parquet
!ls -lh /kaggle/working/*.zip

## 4. RoBERTa (#10), in the order of the issue. `indist` first (its model is reused), then LOGO, paraphrase, cross-domain.

In [ ]:
!python -m src.train_roberta --setting indist

In [ ]:
!python -m src.train_roberta --setting paraphrase_orig --predict-only --model outputs/roberta/indist/model
!python -m src.train_roberta --setting paraphrase --predict-only --model outputs/roberta/indist/model
!python -m src.train_roberta --setting paraphrase_human --predict-only --model outputs/roberta/indist/model

In [ ]:
for gen in ["chatGPT", "davinci", "cohere", "dolly", "bloomz"]:
    !python -m src.train_roberta --setting logo_{gen} --no-save

In [ ]:
for dom in ["wikipedia", "wikihow", "reddit", "arxiv", "peerread"]:
    !python -m src.train_roberta --setting xdomain_{dom} --no-save

## 5. Metrics with the shared evaluator (`src/eval.py` is on `main`)

In [ ]:
import glob, subprocess, sys
for method in ("ppl", "logrank", "fastdetectgpt", "roberta"):
    for p in sorted(glob.glob(f"results/{method}/*/preds.csv")):
        r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
        print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")

## 6. Final zip: everything to commit, in one file

In [ ]:
!zip -rq /kaggle/working/all_results.zip results/ppl results/logrank results/fastdetectgpt results/roberta results/paraphrase_quality.json splits/paraphrase.json splits/paraphrase_human.json outputs/paraphrase/paraphrased.parquet
!ls -lh /kaggle/working/*.zip